# 03 — Match outcome prediction (HOME / DRAW / AWAY)

A 3-class classification model that returns the **probability** of a home win, a draw, or an away win.

- **Data**: `data/training_features.csv` (matches 2010–2026, features already built as differentials).
- **Model**: `HistGradientBoostingClassifier` (gradient boosting, handles missing values natively — no imputation needed).
- **Validation**: **temporal** split (train on the past, test on the recent period).
- **Output**: a `predict(home, away, neutral)` function to test any matchup.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss, accuracy_score, confusion_matrix

DATA = Path("../data/training_features.csv")
df = pd.read_csv(DATA, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
print(f"{len(df):,} matches  |  {df.date.min().date()} -> {df.date.max().date()}")
df["result"].value_counts(normalize=True).round(3)

15,870 matches  |  2010-01-02 -> 2026-06-23


result
HOME    0.479
AWAY    0.289
DRAW    0.232
Name: proportion, dtype: float64

## 1. Features and target

We use the numeric columns describing both teams (strength, form, value, head-to-head...). We **exclude the raw team names**: the model should learn strength-vs-strength logic, not "this specific team wins", so it generalizes well to World Cup matchups.

In [2]:
FEATURES = [
    "neutral",
    "home_ranking", "away_ranking", "ranking_diff",
    "home_elo", "away_elo", "elo_diff",
    "home_squad_age", "away_squad_age",
    "home_market_value", "away_market_value", "market_value_ratio",
    "home_form_win_rate", "home_form_pts_rate", "home_form_gf", "home_form_ga", "home_form_gd",
    "away_form_win_rate", "away_form_pts_rate", "away_form_gf", "away_form_ga", "away_form_gd",
    "h2h_matches", "h2h_home_wins", "h2h_away_wins", "h2h_draws",
    "home_days_rest", "away_days_rest", "is_world_cup",
]
X = df[FEATURES]
y = df["result"]
print("features:", len(FEATURES))

features: 29


## 2. Temporal split

We cut at the 85th percentile date: train on the oldest matches, test on the most recent ones. This simulates a real prediction of the future (no data leakage).

In [3]:
cut = df["date"].quantile(0.85)
is_train = df["date"] <= cut

X_train, y_train = X[is_train], y[is_train]
X_test,  y_test  = X[~is_train], y[~is_train]
print(f"cutoff : {cut.date()}")
print(f"train : {len(X_train):,}   |   test : {len(X_test):,}")

cutoff : 2024-03-22
train : 13,499   |   test : 2,371


## 3. Training

In [4]:
model = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.05,
    l2_regularization=1.0,
    random_state=42,
)
model.fit(X_train, y_train)
print("classes (probability order):", list(model.classes_))

classes (probability order): ['AWAY', 'DRAW', 'HOME']


## 4. Evaluation

We look at the **log-loss** (probability quality, lower = better) compared to a **baseline** (class proportions from the train set), the accuracy, and the confusion matrix.

In [5]:
labels = list(model.classes_)          # ['AWAY', 'DRAW', 'HOME']
proba = model.predict_proba(X_test)
pred  = model.predict(X_test)

ll  = log_loss(y_test, proba, labels=labels)
acc = accuracy_score(y_test, pred)

base_rates = y_train.value_counts(normalize=True)
base = np.tile([base_rates[l] for l in labels], (len(y_test), 1))
base_ll = log_loss(y_test, base, labels=labels)

print(f"model log-loss    : {ll:.4f}")
print(f"baseline log-loss : {base_ll:.4f}")
print(f"accuracy          : {acc:.3f}")

model log-loss    : 0.9003
baseline log-loss : 1.0503
accuracy          : 0.585


In [6]:
cm = confusion_matrix(y_test, pred, labels=labels)
pd.DataFrame(cm, index=[f"true {l}" for l in labels], columns=[f"pred {l}" for l in labels])

,pred AWAY,pred DRAW,pred HOME
true AWAY,393,21,263
true DRAW,191,22,343
true HOME,141,26,971


**Quick calibration check** — for each predicted home-win probability bucket, we check that the actual frequency follows.

In [7]:
home_idx = labels.index("HOME")
p_home = proba[:, home_idx]
actual_home = (y_test.values == "HOME").astype(int)

bins = pd.cut(p_home, [0, .2, .4, .6, .8, 1.0])
cal = pd.DataFrame({"p_pred": p_home, "actual": actual_home}).groupby(bins, observed=True).agg(
    n=("actual", "size"), mean_predicted_prob=("p_pred", "mean"), actual_freq=("actual", "mean")).round(3)
cal

,n,mean_predicted_prob,actual_freq
"(0.0, 0.2]",297,0.129,0.128
"(0.2, 0.4]",584,0.301,0.267
"(0.4, 0.6]",628,0.499,0.479
"(0.6, 0.8]",639,0.699,0.706
"(0.8, 1.0]",223,0.851,0.861


## 5. Feature importance (permutation)

In [8]:
from sklearn.inspection import permutation_importance
imp = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=42, scoring="neg_log_loss")
pd.Series(imp.importances_mean, index=FEATURES).sort_values(ascending=False).head(12).round(4)

market_value_ratio    0.0392
h2h_away_wins         0.0267
h2h_home_wins         0.0180
ranking_diff          0.0104
home_form_ga          0.0080
home_market_value     0.0039
neutral               0.0036
elo_diff              0.0033
away_form_ga          0.0031
home_form_gd          0.0031
away_form_gd          0.0031
home_ranking          0.0029
dtype: float64

## 6. Prediction function

`predict(home, away, neutral=False, is_world_cup=False)`:
1. fetches each team's latest known stats (elo, ranking, value, form...),
2. computes the head-to-head history (h2h),
3. returns the probabilities for **home win / draw / away win**.

In [9]:
def _long_table(df):
    cols = ["ranking", "elo", "squad_age", "market_value",
            "form_win_rate", "form_pts_rate", "form_gf", "form_ga", "form_gd"]
    out = []
    for side in ("home", "away"):
        sub = df[["date", f"{side}_team"] + [f"{side}_{c}" for c in cols]].copy()
        sub.columns = ["date", "team"] + cols
        out.append(sub)
    return pd.concat(out).sort_values("date")

_LT = _long_table(df)

def _team_state(team):
    sub = _LT[_LT["team"] == team]
    if sub.empty:
        raise ValueError(f"Unknown team: {team!r}")
    return sub.iloc[-1]

def _h2h(home, away):
    m = df[((df.home_team == home) & (df.away_team == away)) |
           ((df.home_team == away) & (df.away_team == home))]
    hw = ((m.home_team == home) & (m.result == "HOME")).sum() + ((m.away_team == home) & (m.result == "AWAY")).sum()
    aw = ((m.home_team == away) & (m.result == "HOME")).sum() + ((m.away_team == away) & (m.result == "AWAY")).sum()
    return len(m), int(hw), int(aw), int((m.result == "DRAW").sum())

In [10]:
def _safe_diff(a, b):
    return a - b if pd.notna(a) and pd.notna(b) else np.nan

def predict(home, away, neutral=False, is_world_cup=False):
    """Outcome probabilities for home vs away.
    neutral : True if neutral venue. Returns a dict of probabilities."""
    h, a = _team_state(home), _team_state(away)
    n, hw, aw, dr = _h2h(home, away)
    rest = int(df["home_days_rest"].median())

    row = {
        "neutral": int(neutral),
        "home_ranking": h.ranking, "away_ranking": a.ranking, "ranking_diff": _safe_diff(h.ranking, a.ranking),
        "home_elo": h.elo, "away_elo": a.elo, "elo_diff": _safe_diff(h.elo, a.elo),
        "home_squad_age": h.squad_age, "away_squad_age": a.squad_age,
        "home_market_value": h.market_value, "away_market_value": a.market_value,
        "market_value_ratio": (h.market_value / a.market_value
                               if pd.notna(h.market_value) and pd.notna(a.market_value) and a.market_value else np.nan),
        "home_form_win_rate": h.form_win_rate, "home_form_pts_rate": h.form_pts_rate,
        "home_form_gf": h.form_gf, "home_form_ga": h.form_ga, "home_form_gd": h.form_gd,
        "away_form_win_rate": a.form_win_rate, "away_form_pts_rate": a.form_pts_rate,
        "away_form_gf": a.form_gf, "away_form_ga": a.form_ga, "away_form_gd": a.form_gd,
        "h2h_matches": n, "h2h_home_wins": hw, "h2h_away_wins": aw, "h2h_draws": dr,
        "home_days_rest": rest, "away_days_rest": rest, "is_world_cup": int(is_world_cup),
    }
    p = model.predict_proba(pd.DataFrame([row])[FEATURES])[0]
    d = dict(zip(model.classes_, p))
    return {
        f"{home} (home win)": round(float(d["HOME"]), 3),
        "draw": round(float(d["DRAW"]), 3),
        f"{away} (away win)": round(float(d["AWAY"]), 3),
    }

### Examples

In [11]:
predict("France", "Brazil", neutral=True, is_world_cup=True)

{'France (home win)': 0.546, 'draw': 0.185, 'Brazil (away win)': 0.269}

In [12]:
predict("Argentina", "Saudi Arabia", neutral=True, is_world_cup=True)

{'Argentina (home win)': 0.786, 'draw': 0.07, 'Saudi Arabia (away win)': 0.144}

In [13]:
predict("Germany", "Spain", neutral=False)

{'Germany (home win)': 0.447, 'draw': 0.225, 'Spain (away win)': 0.328}

## 7. Save the model

Persist the trained model + feature list so the dashboard (and other scripts) can load it without retraining.

In [14]:
import joblib
# Refit on the FULL dataset (train+test) for deployment — we keep more recent
# matches in the saved model. (The temporal split above was only for evaluation.)
final_model = HistGradientBoostingClassifier(
    max_iter=400, learning_rate=0.05, l2_regularization=1.0, random_state=42,
).fit(X, y)

out = Path("../models/wc_model.pkl")
out.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"model": final_model, "features": FEATURES}, out)
print("saved ->", out.resolve())

saved -> /sessions/awesome-gracious-shannon/mnt/world_cup_prediction/models/wc_model.pkl
